In [1]:
# VFL SHAP - Multi-Class Version
# All imports at the top
import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Import utility functions
from vfl_utils import (
    simplify_label,
    categorize_feature_by_evidence,
    format_action_readable,
    ATTACK_ACTIONS,
    split_features_balanced,
    get_evidence_type,
    generate_party_name,
    generate_domain,
    generate_action,
    get_party_actions_for_attack
)

# -----------------------------


In [2]:
# 1. Load Dataset & Extract Multi-Class Labels
# -----------------------------
df = pd.read_csv("D:\\Projects\\Datasets\\undersampled_CIC2017_dataset.csv")
df = df.drop(columns=["Flow ID", "Src IP", "Dst IP", "Timestamp"], errors="ignore")

# Print dataset information
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")


# Extract and simplify label information
label_col = "label"

# Group similar attack types
# Apply simplification
df['label_simplified'] = df[label_col].apply(simplify_label)

# Group classes with less than 200 rows into "OTHERS"
label_counts = df['label_simplified'].value_counts()
min_rows = 200

# Find labels with less than min_rows
small_labels = label_counts[label_counts < min_rows].index.tolist()

if len(small_labels) > 0:
    print(f"Grouping {len(small_labels)} classes with < {min_rows} rows into 'OTHERS':")
    for small_label in small_labels:
        count = label_counts[small_label]
        print(f"  {small_label}: {count} rows -> OTHERS")
    
    # Replace small labels with "OTHERS"
    df.loc[df['label_simplified'].isin(small_labels), 'label_simplified'] = 'OTHERS'

# Create mapping to numeric
unique_labels = sorted(df['label_simplified'].unique())
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
label_names_short = {idx: label for label, idx in label_mapping.items()}

df['label_numeric'] = df['label_simplified'].map(label_mapping)
label_col = 'label_numeric'
num_classes = len(unique_labels)

print(f"Found {num_classes} simplified label types:")
for orig_label in sorted(df['label_simplified'].unique()):
    num = label_mapping[orig_label]
    print(f"  {orig_label} -> {num}")

label_mapping_dict = label_names_short

print(f"\nNumber of classes: {num_classes}")
print(f"Label distribution:\n{df[label_col].value_counts().sort_index()}")
# -----------------------------

Total rows: 390398
Total columns: 89
Grouping 2 classes with < 200 rows into 'OTHERS':
  INFILTRATION: 122 rows -> OTHERS
  HEARTBLEED: 70 rows -> OTHERS
Found 9 simplified label types:
  BENIGN -> 0
  BOT -> 1
  DDOS -> 2
  DOS -> 3
  FTPPATATOR -> 4
  OTHERS -> 5
  PORTSCAN -> 6
  SSHPATATOR -> 7
  WEBATTACK -> 8

Number of classes: 9
Label distribution:
label_numeric
0    300332
1      1228
2     12582
3     44020
4      3974
5       192
6     20525
7      2978
8      4567
Name: count, dtype: int64


In [3]:
# 2. Evidence-Type Vertical Feature Partition (IDS-Optimized with Balanced Split)
# -----------------------------
# vfl_utils imports are in Cell 0

non_feature_cols = ["Flow ID", "Src IP", "Dst IP", "Timestamp", "label", "label_numeric", "label_simplified"]
all_features = [col for col in df.columns if col not in non_feature_cols]

# Filter to only numeric columns (exclude string/object columns)
all_features = [col for col in all_features if df[col].dtype in ['int64', 'float64', 'int32', 'float32']]

print(f"Total features: {len(all_features)}")

# Split features into parties using balanced splitting function
# This ensures more even distribution while preserving evidence type grouping
# Minimum 20 features per party to ensure meaningful VFL learning
party1_features, party2_features, party3_features, feature_categories = split_features_balanced(
    all_features, 
    num_parties=3, 
    min_features_per_party=20,  # Ensure at least 20 features per party
    balance_threshold=0.15,  # Allow up to 15% difference between parties
    random_seed=42
)

# Display feature distribution
print(f"\nFeature categories by evidence type:")
for cat, feats in feature_categories.items():
    print(f"  {cat}: {len(feats)} features")

print(f"\nParty 1: {len(party1_features)} features")
print(f"Party 2: {len(party2_features)} features")
print(f"Party 3: {len(party3_features)} features")

# Check balance
party_sizes = [len(party1_features), len(party2_features), len(party3_features)]
total_features = sum(party_sizes)
target_size = total_features / 3
balance_ratio = (max(party_sizes) - min(party_sizes)) / target_size if target_size > 0 else 0
print(f"\nBalance check:")
print(f"  Target size per party: {target_size:.1f}")
print(f"  Actual sizes: {party_sizes}")
print(f"  Balance ratio: {balance_ratio:.2%} (lower is better, <15% is good)")

# IDS-style attack-type action mapping

# Helper function to format actions in human-readable way
# Generate party names, domains, and actions based on evidence type




party_names = [
    generate_party_name(party1_features, 1),
    generate_party_name(party2_features, 2),
    generate_party_name(party3_features, 3)
]

party_feature_groups = [
    f"Features: {', '.join(party1_features[:3])}..." if len(party1_features) > 3 else f"Features: {', '.join(party1_features)}",
    f"Features: {', '.join(party2_features[:3])}..." if len(party2_features) > 3 else f"Features: {', '.join(party2_features)}",
    f"Features: {', '.join(party3_features[:3])}..." if len(party3_features) > 3 else f"Features: {', '.join(party3_features)}"
]

party_domains = [
    generate_domain(party1_features, 1),
    generate_domain(party2_features, 2),
    generate_domain(party3_features, 3)
]

party_actions = [
    generate_action(party1_features, 1),
    generate_action(party2_features, 2),
    generate_action(party3_features, 3)
]

# Create comprehensive action mapping per party for SHAP analysis
# get_party_actions_for_attack is now imported from vfl_utils

# Store action mapping for later use
party_action_mapping = {}  # Will be: {party_idx: {attack_type: action}}
party_feature_lists = [party1_features, party2_features, party3_features]
for i in range(3):
    party_action_mapping[i] = {}
    for attack_type in ATTACK_ACTIONS.keys():
        party_action_mapping[i][attack_type] = get_party_actions_for_attack(party_feature_lists[i], attack_type)

print("\n=== Party Configuration ===")
for i in range(3):
    features_list = [party1_features, party2_features, party3_features][i]
    evidence_type = get_evidence_type(features_list)
    print(f"\n{party_names[i]}:")
    print(f"  Domain: {party_domains[i]}")
    print(f"  Evidence Type: {evidence_type}")
    print(f"  Features ({len(features_list)}): {features_list[:5]}")
    print(f"  Primary Actions: {party_actions[i]}")
    print(f"  All Attack Actions:")
    for attack_type, action in party_action_mapping[i].items():
        if attack_type in ['DDOS', 'DOS', 'PORTSCAN', 'WEBATTACK', 'SSHPATATOR', 'FTPPATATOR']:
            print(f"    {attack_type}: {action}")
# -----------------------------

Total features: 88

Feature categories by evidence type:
  evidence_volume_rate: 23 features
  evidence_packet_size: 27 features
  evidence_timing_direction: 38 features

Party 1: 23 features
Party 2: 27 features
Party 3: 38 features

Balance check:
  Target size per party: 29.3
  Actual sizes: [23, 27, 38]
  Balance ratio: 51.14% (lower is better, <15% is good)

=== Party Configuration ===

Volume_Rate_Sensor_Party1:
  Domain: Volume & Rate Analysis (DoS/DDoS Detection)
  Evidence Type: evidence_volume_rate
  Features (23): ['udps.srcdst_rst_packet_count', 'udps.srcdst_tcp_packet_count', 'bidirectional_duration_ms', 'udps.udp_packet_count', 'udps.srcdst_psh_packet_count']
  Primary Actions: DDOS: rate-limit, SYN cookies, WAF rules, drop bursts, auto-scale, block top talkers; DOS: rate-limit, SYN cookies, WAF rules, drop bursts, auto-scale, block top talkers
  All Attack Actions:
    DDOS: rate-limit, SYN cookies, WAF rules, drop bursts, auto-scale, block top talkers
    DOS: rate-limi

In [4]:
# 3. Preprocess Features
# -----------------------------
scaler1 = StandardScaler()
scaler2 = StandardScaler()
scaler3 = StandardScaler()

X1 = torch.tensor(scaler1.fit_transform(df[party1_features].values), dtype=torch.float32)
X2 = torch.tensor(scaler2.fit_transform(df[party2_features].values), dtype=torch.float32)
X3 = torch.tensor(scaler3.fit_transform(df[party3_features].values), dtype=torch.float32)

# Ensure we use numeric labels
if 'label_numeric' in df.columns:
    y = torch.tensor(df['label_numeric'].values, dtype=torch.long)  # long for multi-class
else:
    # Fallback: convert label_col to numeric if needed
    if df[label_col].dtype == 'object' or df[label_col].dtype == 'string':
        # Convert string labels to numeric
        unique_labels = sorted(df[label_col].unique())
        label_to_num = {label: idx for idx, label in enumerate(unique_labels)}
        y = torch.tensor(df[label_col].map(label_to_num).values, dtype=torch.long)
    else:
        y = torch.tensor(df[label_col].values, dtype=torch.long)

x_parts = [X1, X2, X3]
# -----------------------------

In [5]:
# 4. Train/Val/Test Split (Improved Stratification)
# -----------------------------
# Use numeric labels for stratification
stratify_labels = df['label_numeric'] if 'label_numeric' in df.columns else y.numpy()

# Check class distribution before split
print("Class distribution before split:")
print(stratify_labels.value_counts().sort_index())

# First split: train+val (80%) vs test (20%)
trainval_idx, test_idx = train_test_split(
    range(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=stratify_labels
)

# Second split: train (64%) vs val (16%) from train+val
trainval_labels = stratify_labels.iloc[trainval_idx] if isinstance(stratify_labels, pd.Series) else stratify_labels[trainval_idx]
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=0.2,  # 20% of 80% = 16% of total
    random_state=42,
    stratify=trainval_labels
)

# Create splits
x_train_parts = [x[train_idx] for x in x_parts]
x_val_parts = [x[val_idx] for x in x_parts]
x_test_parts = [x[test_idx] for x in x_parts]
y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

# Verify stratification
print("\nTrain set class distribution:")
train_labels = stratify_labels.iloc[train_idx] if isinstance(stratify_labels, pd.Series) else stratify_labels[train_idx]
print(pd.Series(train_labels).value_counts().sort_index())

print("\nValidation set class distribution:")
val_labels = stratify_labels.iloc[val_idx] if isinstance(stratify_labels, pd.Series) else stratify_labels[val_idx]
print(pd.Series(val_labels).value_counts().sort_index())

print("\nTest set class distribution:")
test_labels = stratify_labels.iloc[test_idx] if isinstance(stratify_labels, pd.Series) else stratify_labels[test_idx]
print(pd.Series(test_labels).value_counts().sort_index())

print(f"\nSplit sizes: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")
# -----------------------------

Class distribution before split:
label_numeric
0    300332
1      1228
2     12582
3     44020
4      3974
5       192
6     20525
7      2978
8      4567
Name: count, dtype: int64

Train set class distribution:
label_numeric
0    192212
1       786
2      8053
3     28173
4      2543
5       123
6     13136
7      1905
8      2923
Name: count, dtype: int64

Validation set class distribution:
label_numeric
0    48053
1      196
2     2013
3     7043
4      636
5       31
6     3284
7      477
8      731
Name: count, dtype: int64

Test set class distribution:
label_numeric
0    60067
1      246
2     2516
3     8804
4      795
5       38
6     4105
7      596
8      913
Name: count, dtype: int64

Split sizes: Train=249854, Val=62464, Test=78080


In [6]:
# 5. Multi-Class VFL Model (Improved Architecture)
# -----------------------------
class LocalEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=64, hidden_dim=128):
        super().__init__()
        # Deeper encoder for better feature learning
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        return self.net(x)


class ActiveClassifier(nn.Module):
    def __init__(self, embed_dim=64, num_classes=5, hidden_dim=128):
        super().__init__()
        # Deeper classifier
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, h):
        # Return logits (NO softmax) - CrossEntropyLoss applies softmax internally
        return self.net(h)


class VFLModel(nn.Module):
    def __init__(self, input_dims, embed_dim=64, num_classes=5, hidden_dim=128):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_classes = num_classes
        self.encoders = nn.ModuleList([LocalEncoder(dim, embed_dim, hidden_dim) for dim in input_dims])
        # Improved fusion: concat embeddings instead of sum
        fusion_dim = embed_dim * len(input_dims)  # 3 parties * embed_dim
        self.classifier = ActiveClassifier(fusion_dim, num_classes, hidden_dim)

    def forward(self, x_parts):
        embeddings = [enc(x) for x, enc in zip(x_parts, self.encoders)]
        # Improved fusion: concatenate instead of sum for better separation
        h = torch.cat(embeddings, dim=1)  # [B, embed_dim*3] instead of [B, embed_dim]
        y_hat = self.classifier(h)
        return y_hat

    def get_party_embeddings(self, x_parts):
        self.eval()
        with torch.no_grad():
            embeddings = [enc(x) for x, enc in zip(x_parts, self.encoders)]
        return embeddings
# -----------------------------

In [7]:
# 6. Train & Evaluate Functions (Multi-Class with Validation & Early Stopping)
# -----------------------------
def evaluate_model(model, x_parts, y, criterion=None):
    """Evaluate model and return loss, accuracy, recall, f1"""
    model.eval()
    with torch.no_grad():
        y_hat = model(x_parts)
        y_pred = torch.argmax(y_hat, dim=1)
        
        loss = None
        if criterion is not None:
            loss = criterion(y_hat, y.long()).item()
        
        acc = accuracy_score(y.cpu(), y_pred.cpu())
        rec = recall_score(y.cpu(), y_pred.cpu(), average='macro', zero_division=0)
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='macro', zero_division=0)
        
    return loss, acc, rec, f1

def train_vfl(model, x_train_parts, y_train, x_val_parts, y_val, 
              epochs=100, lr=1e-3, use_class_weights=True, 
              early_stop_patience=20, min_delta=0.001, eval_every=5):
    """
    Train VFL model with validation, early stopping, and best model checkpointing
    
    Args:
        model: VFL model to train
        x_train_parts: Training features for each party
        y_train: Training labels
        x_val_parts: Validation features for each party
        y_val: Validation labels
        epochs: Maximum epochs
        lr: Learning rate
        use_class_weights: Use class weights for imbalanced data
        early_stop_patience: Early stopping patience (epochs)
        min_delta: Minimum improvement to reset patience
        eval_every: Evaluate validation every N epochs
    """
    # Calculate class weights for imbalanced data (clamped to prevent extreme values)
    if use_class_weights:
        y_np = y_train.cpu().numpy()
        class_counts = np.bincount(y_np, minlength=model.num_classes)
        total = len(y_np)
        class_weights = total / (len(class_counts) * class_counts.astype(float))
        # Clamp weights to prevent extreme values (max weight = 20)
        class_weights = np.clip(class_weights, None, 20.0)
        class_weights = torch.tensor(class_weights, dtype=torch.float32)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print(f"Using class weights (clamped max=20): {class_weights.numpy()}")
    else:
        criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    # Scheduler uses validation loss
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=15, 
        threshold=0.0001, min_lr=1e-6
    )
    
    # Early stopping tracking
    best_val_f1 = -1.0
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    best_model_state = None
    
    print(f"\nStarting training with early stopping (patience={early_stop_patience}, min_delta={min_delta})")
    print(f"Evaluating validation every {eval_every} epochs\n")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        y_hat_train = model(x_train_parts)
        train_loss = criterion(y_hat_train, y_train.long())
        
        optimizer.zero_grad()
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Validation phase (every eval_every epochs or at start/end)
        if epoch % eval_every == 0 or epoch == 0 or epoch == epochs - 1:
            val_loss, val_acc, val_rec, val_f1 = evaluate_model(model, x_val_parts, y_val, criterion)
            
            # Update scheduler with validation loss (fix warning by using .item())
            scheduler.step(val_loss)
            current_lr = optimizer.param_groups[0]['lr']
            
            # Check for improvement
            improved = False
            if val_f1 > best_val_f1 + min_delta:
                improved = True
                best_val_f1 = val_f1
                best_val_loss = val_loss
                best_epoch = epoch
                patience_counter = 0
                # Save best model state
                best_model_state = model.state_dict().copy()
                print(f"[VFL] Epoch {epoch:3d} | Train Loss: {train_loss.item():.4f} | "
                      f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | "
                      f"Val Rec: {val_rec:.4f} | LR: {current_lr:.6f} | ✓ BEST")
            else:
                patience_counter += 1
                print(f"[VFL] Epoch {epoch:3d} | Train Loss: {train_loss.item():.4f} | "
                      f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | "
                      f"Val Rec: {val_rec:.4f} | LR: {current_lr:.6f} | "
                      f"Patience: {patience_counter}/{early_stop_patience}")
            
            # Early stopping check
            if patience_counter >= early_stop_patience:
                print(f"\n[EARLY STOP] No improvement for {early_stop_patience} epochs. "
                      f"Best F1: {best_val_f1:.4f} at epoch {best_epoch}")
                break
        
        # Print training progress (every 10 epochs if not evaluating)
        elif epoch % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"[VFL] Epoch {epoch:3d} | Train Loss: {train_loss.item():.4f} | LR: {current_lr:.6f}")
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n[CHECKPOINT] Loaded best model from epoch {best_epoch} (Val F1: {best_val_f1:.4f})")
    
    return best_epoch, best_val_f1, best_val_loss

def evaluate(model, x_parts, y, print_per_class=False):
    """Comprehensive evaluation with per-class metrics"""
    model.eval()
    with torch.no_grad():
        y_hat = model(x_parts)
        y_pred = torch.argmax(y_hat, dim=1)
    
    acc = accuracy_score(y.cpu(), y_pred.cpu())
    rec_macro = recall_score(y.cpu(), y_pred.cpu(), average='macro', zero_division=0)
    rec_weighted = recall_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
    f1_macro = f1_score(y.cpu(), y_pred.cpu(), average='macro', zero_division=0)
    f1_weighted = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
    
    print(f"\n=== Evaluation Results ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro Recall: {rec_macro:.4f} | Weighted Recall: {rec_weighted:.4f}")
    print(f"Macro F1: {f1_macro:.4f} | Weighted F1: {f1_weighted:.4f}")
    
    if print_per_class:
        print(f"\n=== Per-Class Metrics ===")
        class_names = [label_mapping_dict.get(i, f"Class_{i}") for i in range(model.num_classes)]
        print(classification_report(y.cpu(), y_pred.cpu(), target_names=class_names, zero_division=0))
    
    return acc, rec_macro, f1_macro
# -----------------------------

In [ ]:
# 7. Train VFL Model with Validation & Early Stopping
# -----------------------------
embed_dim = 64
hidden_dim = 128
model = VFLModel(
    input_dims=[len(party1_features), len(party2_features), len(party3_features)],
    embed_dim=embed_dim,
    num_classes=num_classes,
    hidden_dim=hidden_dim
)

print(f"Model parameters: embed_dim={embed_dim}, hidden_dim={hidden_dim}, num_classes={num_classes}")
print(f"Fusion: concat (embed_dim*3 = {embed_dim*3})")
print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

# Train with validation and early stopping
best_epoch, best_val_f1, best_val_loss = train_vfl(
    model, x_train_parts, y_train, x_val_parts, y_val,
    epochs=20, lr=0.002, use_class_weights=True,
    early_stop_patience=20, min_delta=0.001, eval_every=5
)

# Final evaluation on test set
print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)
acc, rec, f1 = evaluate(model, x_test_parts, y_test, print_per_class=True)

# Confusion matrix
y_test_pred = torch.argmax(model(x_test_parts), dim=1).cpu().numpy()
y_test_np = y_test.cpu().numpy()
cm = confusion_matrix(y_test_np, y_test_pred)
print(f"\n=== Confusion Matrix ===")
print("Rows = True labels, Columns = Predicted labels")
class_names = [label_mapping_dict.get(i, f"Class_{i}") for i in range(num_classes)]
print("Classes:", class_names)
print(cm)

# Save performance metrics
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
perf_df = pd.DataFrame({
    "Metric": ["Accuracy", "Macro_Recall", "Macro_F1", "Best_Val_F1", "Best_Epoch"],
    "Value": [acc, rec, f1, best_val_f1, best_epoch]
})
perf_filename = f"outputs/vfl_shap_performance_{timestamp}.csv"
perf_df.to_csv(perf_filename, index=False)
print(f"\nPerformance saved to {perf_filename}")
# -----------------------------

Model parameters: embed_dim=64, hidden_dim=128, num_classes=9
Fusion: concat (embed_dim*3 = 192)
Train: 249854, Val: 62464, Test: 78080
Using class weights (clamped max=20): [ 0.14443196 20.          3.4473557   0.9853958  10.916852   20.
  2.113395   14.572995    9.497624  ]

Starting training with early stopping (patience=20, min_delta=0.001)
Evaluating validation every 5 epochs

[VFL] Epoch   0 | Train Loss: 2.2042 | Val Loss: 2.1141 | Val F1: 0.0749 | Val Rec: 0.2459 | LR: 0.002000 | ✓ BEST
[VFL] Epoch   5 | Train Loss: 1.7209 | Val Loss: 1.5730 | Val F1: 0.4671 | Val Rec: 0.6423 | LR: 0.002000 | ✓ BEST


In [ ]:
# 8. Build Party-Level Meta-Features (Full Embeddings)
# -----------------------------
# Use FULL party embeddings instead of mean to preserve information
# This improves meta-model learning significantly
model.eval()
with torch.no_grad():
    train_embeds = model.get_party_embeddings(x_train_parts)  # list of (N, embed_dim=64)
    test_embeds = model.get_party_embeddings(x_test_parts)     # list of (N, embed_dim=64)

# Concatenate full embeddings: (N, 64) + (N, 64) + (N, 64) = (N, 192)
# This preserves all information instead of collapsing to 3 scalars
X_train_meta = torch.cat([train_embeds[0], train_embeds[1], train_embeds[2]], dim=1)  # (N, 192)
X_test_meta = torch.cat([test_embeds[0], test_embeds[1], test_embeds[2]], dim=1)        # (N, 192)

print(f"Meta-features shape: Train={X_train_meta.shape}, Test={X_test_meta.shape}")
print(f"Using full embeddings (192 dims) instead of mean (3 dims) for better meta-model learning")
# -----------------------------

In [ ]:
# 9. Multi-Class Meta-Model (Improved: MLP + Soft Distillation)
# -----------------------------
# Upgraded meta-model: MLP architecture + soft distillation with temperature
# This significantly improves agreement between VFL model and meta-model

class PartyMetaModel(nn.Module):
    def __init__(self, in_dim=192, num_classes=9, hidden_dim=128):
        super().__init__()
        # MLP instead of linear: can represent complex decision boundaries
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x_meta):
        # Return logits (NO softmax) - soft distillation uses logits
        return self.net(x_meta)

# Initialize meta-model with full embedding dimension (192)
meta_model = PartyMetaModel(in_dim=X_train_meta.shape[1], num_classes=num_classes, hidden_dim=128)
print(f"Meta-model input dim: {X_train_meta.shape[1]}, output classes: {num_classes}")

# Soft distillation: KL divergence with temperature (preserves teacher confidence)
T = 3.0  # Temperature for softmax smoothing
kl_loss = nn.KLDivLoss(reduction="batchmean")
optimizer = optim.Adam(meta_model.parameters(), lr=1e-3, weight_decay=1e-4)
meta_epochs = 50

print(f"\nTraining meta-model with soft distillation (T={T})...")
model.eval()
for epoch in range(meta_epochs):
    meta_model.train()
    optimizer.zero_grad()
    
    # Teacher: get logits from VFL model
    with torch.no_grad():
        teacher_logits = model(x_train_parts)  # (N, num_classes) logits
        # Soft targets: apply temperature to teacher logits
        teacher_probs_T = torch.softmax(teacher_logits / T, dim=1)
    
    # Student: get logits from meta-model
    student_logits = meta_model(X_train_meta)  # (N, num_classes) logits
    # Soft predictions: apply temperature and log
    student_log_probs_T = torch.log_softmax(student_logits / T, dim=1)
    
    # KL divergence loss (scaled by T^2 for proper gradient scaling)
    loss = kl_loss(student_log_probs_T, teacher_probs_T) * (T * T)
    
    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"[META] Epoch {epoch} | Distillation Loss: {loss.item():.4f}")

# Meta-model fidelity evaluation
print(f"\n=== Meta-Model Fidelity Evaluation ===")
meta_model.eval()
with torch.no_grad():
    # Get predictions from both models
    teacher_test = model(x_test_parts)              # (N, num_classes) logits
    student_test = meta_model(X_test_meta)         # (N, num_classes) logits
    
    # Hard predictions (argmax)
    teacher_pred = teacher_test.argmax(dim=1).cpu().numpy()
    student_pred = student_test.argmax(dim=1).cpu().numpy()
    
    # Agreement metric (accuracy between teacher and student predictions)
    meta_acc = accuracy_score(teacher_pred, student_pred)
    
    # Additional metrics: KL divergence and logit MSE
    teacher_probs_test = torch.softmax(teacher_test, dim=1)
    student_probs_test = torch.softmax(student_test, dim=1)
    kl_div = kl_loss(torch.log(student_probs_test + 1e-8), teacher_probs_test).item()
    logit_mse = torch.mean((teacher_test - student_test) ** 2).item()

print(f"[META] Agreement (Meta vs VFL): {meta_acc:.4f}")
print(f"[META] KL Divergence: {kl_div:.4f}")
print(f"[META] Logit MSE: {logit_mse:.4f}")

if meta_acc > 0.70:
    print(f" Good agreement! Meta-model successfully mimics VFL model.")
elif meta_acc > 0.50:
    print(f"Moderate agreement. Consider increasing meta-model capacity or training epochs.")
else:
    print(f"Low agreement. Check if VFL model is stable or increase meta-model capacity.")
# -----------------------------

In [ ]:
# 10. SHAP on Party Meta-Features (Full Embeddings)
# -----------------------------
# Note: Meta-features are now 192-dim (64 per party), so we compute SHAP on all 192 dims
# then aggregate per party to get party-level attributions

meta_model.eval()

bg_size = min(100, X_train_meta.shape[0])
bg_idx = torch.randperm(X_train_meta.shape[0])[:bg_size]
background = X_train_meta[bg_idx].detach().cpu().numpy()

def meta_predict(x_np):
    x_t = torch.tensor(x_np, dtype=torch.float32)
    with torch.no_grad():
        logits = meta_model(x_t)
        # Apply softmax to get probabilities for SHAP
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    return probs

print(f"Computing SHAP on {X_train_meta.shape[1]}-dim meta-features (64 dims per party)...")
explainer = shap.KernelExplainer(meta_predict, background)

n_explain = min(200, X_test_meta.shape[0])
X_explain = X_test_meta[:n_explain].detach().cpu().numpy()

# Compute SHAP values (will be 192-dim per sample)
shap_values = explainer.shap_values(X_explain, nsamples=200)

# Get predictions for selecting correct SHAP values
y_meta_pred_explain = torch.argmax(meta_model(torch.tensor(X_explain, dtype=torch.float32)), dim=1).cpu().numpy()

# Aggregate 192-dim SHAP values to 3 party-level attributions
# Structure: first 64 dims = Party 1, next 64 dims = Party 2, last 64 dims = Party 3
embed_dim = 64  # Each party has 64-dim embeddings
party_dims = [embed_dim, embed_dim, embed_dim]  # [64, 64, 64]

def aggregate_shap_to_parties(shap_vals, n_samples):
    """
    Aggregate 192-dim SHAP values to 3 party-level values by summing within each party block.
    
    Args:
        shap_vals: SHAP values array, shape can be:
            - List of arrays: [num_classes, n_samples, 192] or [n_samples, 192, num_classes]
            - Single array: [n_samples, 192] or [num_classes, n_samples, 192]
        n_samples: Number of samples
    
    Returns:
        Aggregated SHAP values: [n_samples, 3] (one value per party)
    """
    # Handle list format (multi-class)
    if isinstance(shap_vals, list):
        # For each class, aggregate to parties
        aggregated = []
        for class_idx in range(len(shap_vals)):
            class_shap = shap_vals[class_idx]  # [n_samples, 192]
            if len(class_shap.shape) == 2 and class_shap.shape[1] == embed_dim * 3:
                # Sum within each party block
                party_shap = np.zeros((n_samples, 3))
                for i in range(3):
                    start_idx = i * embed_dim
                    end_idx = (i + 1) * embed_dim
                    party_shap[:, i] = class_shap[:, start_idx:end_idx].sum(axis=1)
                aggregated.append(party_shap)
            else:
                # Fallback: if shape doesn't match, try to reshape
                aggregated.append(class_shap)
        return aggregated
    else:
        # Single array format
        shap_vals = np.asarray(shap_vals)
        if len(shap_vals.shape) == 2:
            # [n_samples, 192] - aggregate directly
            if shap_vals.shape[1] == embed_dim * 3:
                party_shap = np.zeros((n_samples, 3))
                for i in range(3):
                    start_idx = i * embed_dim
                    end_idx = (i + 1) * embed_dim
                    party_shap[:, i] = shap_vals[:, start_idx:end_idx].sum(axis=1)
                return party_shap
        elif len(shap_vals.shape) == 3:
            # [num_classes, n_samples, 192] or [n_samples, 192, num_classes]
            if shap_vals.shape[0] == num_classes and shap_vals.shape[2] == embed_dim * 3:
                # [num_classes, n_samples, 192]
                aggregated = []
                for class_idx in range(num_classes):
                    class_shap = shap_vals[class_idx]  # [n_samples, 192]
                    party_shap = np.zeros((n_samples, 3))
                    for i in range(3):
                        start_idx = i * embed_dim
                        end_idx = (i + 1) * embed_dim
                        party_shap[:, i] = class_shap[:, start_idx:end_idx].sum(axis=1)
                    aggregated.append(party_shap)
                return aggregated
            elif shap_vals.shape[0] == n_samples and shap_vals.shape[1] == embed_dim * 3:
                # [n_samples, 192, num_classes] - transpose first
                shap_vals = shap_vals.transpose(2, 0, 1)  # [num_classes, n_samples, 192]
                aggregated = []
                for class_idx in range(num_classes):
                    class_shap = shap_vals[class_idx]  # [n_samples, 192]
                    party_shap = np.zeros((n_samples, 3))
                    for i in range(3):
                        start_idx = i * embed_dim
                        end_idx = (i + 1) * embed_dim
                        party_shap[:, i] = class_shap[:, start_idx:end_idx].sum(axis=1)
                    aggregated.append(party_shap)
                return aggregated
    
    # Fallback: return as-is if shape doesn't match expected
    print(f"Warning: Unexpected SHAP shape {shap_vals.shape}, returning as-is")
    return shap_vals

# Aggregate SHAP values to party level
shap_values_aggregated = aggregate_shap_to_parties(shap_values, n_explain)

# Handle SHAP output format - aggregated values should already be [n_explain, 3] or list of [n_explain, 3]
if isinstance(shap_values_aggregated, list):
    # Multi-class: SHAP returns list where each element is [n_explain, 3] for that class
    # Select SHAP values for predicted class for each sample
    shap_values_selected = []
    for i in range(n_explain):
        pred_class = int(y_meta_pred_explain[i])
        if pred_class < len(shap_values_aggregated):
            shap_values_selected.append(shap_values_aggregated[pred_class][i])
        else:
            # Fallback: use first class if prediction is out of bounds
            shap_values_selected.append(shap_values_aggregated[0][i])
    shap_values = np.array(shap_values_selected)  # [n_explain, 3]
else:
    # Already aggregated to [n_explain, 3]
    shap_values = np.asarray(shap_values_aggregated)
    if len(shap_values.shape) == 2 and shap_values.shape[1] == 3:
        # Already in correct format [n_explain, 3]
        pass
    elif len(shap_values.shape) == 3:
        # Could be [num_classes, n_explain, 3] - select by predicted class
        if shap_values.shape[0] == num_classes and shap_values.shape[2] == 3:
            shap_values_selected = []
            for i in range(n_explain):
                pred_class = int(y_meta_pred_explain[i])
                if pred_class < shap_values.shape[0]:
                    shap_values_selected.append(shap_values[pred_class, i, :])
                else:
                    shap_values_selected.append(shap_values[0, i, :])
            shap_values = np.array(shap_values_selected)  # [n_explain, 3]
        else:
            print(f"Warning: Unexpected aggregated SHAP shape {shap_values.shape}")
            # Try to extract
            shap_values = shap_values.reshape(-1, 3)[:n_explain]
    else:
        print(f"Warning: Unexpected aggregated SHAP shape {shap_values.shape}, attempting to reshape...")
        shap_values = shap_values.reshape(n_explain, -1)
        if shap_values.shape[1] > 3:
            # Sum to get 3 party values if needed
            party_size = shap_values.shape[1] // 3
            shap_values = shap_values.reshape(n_explain, 3, party_size).sum(axis=2)

print(f"SHAP values final shape: {shap_values.shape} (should be [{n_explain}, 3])")
print(f"Aggregated from 192-dim meta-features to 3 party-level attributions")
# -----------------------------

In [ ]:
# 11. Aggregate SHAP for Multi-Class
# -----------------------------
phi = shap_values  # [n_explain, 3]
phi_abs = np.abs(phi)

# Global mean |SHAP| and percentage per party
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
mean_phi_abs = phi_abs.mean(axis=0)
mean_phi_pct = mean_phi_abs / mean_phi_abs.sum()

print("\n=== Global SHAP Party Attributions ===")
global_rows = []
for i, name in enumerate(party_names):
    m_abs = float(mean_phi_abs[i])
    m_pct = float(mean_phi_pct[i])
    print(f"{name}: mean |SHAP| = {m_abs:.6f}, mean contribution = {m_pct*100:5.2f}%")
    global_rows.append({
        "Party": name,
        "Domain": party_domains[i],
        "Feature_Group": party_feature_groups[i],
        "Mean_abs_SHAP_All": m_abs,
        "Mean_contrib_All": m_pct,
    })
global_df = pd.DataFrame(global_rows)
global_filename = f"outputs/vfl_shap_global_summary_{timestamp}.csv"
global_df.to_csv(global_filename, index=False)

# Per-class SHAP analysis
y_test_np = y_test.cpu().numpy()
y_explain = y_test_np[:n_explain]

print("\n=== SHAP Party Attributions by Class ===")
for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_mask = (y_explain == class_idx)
    
    if class_mask.sum() == 0:
        continue
    
    phi_class = phi_abs[class_mask]
    mean_phi_class = phi_class.mean(axis=0)
    mean_pct_class = mean_phi_class / mean_phi_class.sum()
    
    print(f"\n{class_name} (class {class_idx}):")
    class_rows = []
    for i, name in enumerate(party_names):
        c_abs = float(mean_phi_class[i])
        c_pct = float(mean_pct_class[i])
        print(f"  {name}: mean |SHAP| = {c_abs:.6f}, mean contribution = {c_pct*100:5.2f}%")
        class_rows.append({
            "Party": name,
            "Domain": party_domains[i],
            "Feature_Group": party_feature_groups[i],
            "Class": class_name,
            "Class_Index": class_idx,
            "Mean_abs_SHAP": c_abs,
            "Mean_contrib": c_pct,
        })
    
    class_df = pd.DataFrame(class_rows)
    class_filename = f"summary/vfl_shap_{class_name.lower()}_summary_{timestamp}.csv"
    class_df.to_csv(class_filename, index=False)

# Find dominant party per class
print("\n=== Dominant Party per Class ===")
for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_mask = (y_explain == class_idx)
    
    if class_mask.sum() == 0:
        continue
    
    phi_class = phi_abs[class_mask]
    mean_phi_class = phi_class.mean(axis=0)
    mean_pct_class = mean_phi_class / mean_phi_class.sum()
    
    top_party_idx = int(np.argmax(mean_pct_class))
    top_party_name = party_names[top_party_idx]
    top_party_share = float(mean_pct_class[top_party_idx]) * 100.0
    
    print(f"{class_name}: {top_party_name} ({top_party_share:.2f}%)")
# -----------------------------

In [ ]:
# 12. Agent-Level Mitigation Summary (Attack-Type Specific Actions)
# -----------------------------
agent_rows = []
for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_mask = (y_explain == class_idx)
    
    if class_mask.sum() == 0:
        continue
    
    phi_class = phi_abs[class_mask]
    mean_phi_class = phi_class.mean(axis=0)
    mean_pct_class = mean_phi_class / mean_phi_class.sum()
    
    top_party_idx = int(np.argmax(mean_pct_class))
    
    # Get attack-type specific action for this class
    attack_type = class_name.upper()
    
    for i, name in enumerate(party_names):
        # CORRECT ACTION PLAN: Dominant party gets primary action
        is_dominant = (i == top_party_idx)
        
        if is_dominant:
            # Dominant party gets primary action for this attack type
            suggested_action = ATTACK_ACTIONS.get(attack_type, "monitor and alert")
        else:
            # Non-dominant parties keep their secondary actions (monitor)
            suggested_action = party_action_mapping[i].get(attack_type, party_actions[i])
        
        
        agent_rows.append({
            "Class": class_name,
            "Class_Index": class_idx,
            "Party": name,
            "Domain": party_domains[i],
            "Feature_Group": party_feature_groups[i],
            "Mean_contrib": float(mean_pct_class[i]),
            "Suggested_Action": suggested_action,
            "Is_Dominant": is_dominant
        })

agent_df = pd.DataFrame(agent_rows)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
agent_filename = f"outputs/vfl_shap_agent_mitigation_summary_{timestamp}.csv"
agent_df.to_csv(agent_filename, index=False)
# # print("Mitigation summary saved to: outputs/vfl_shap_agent_mitigation_summary.csv")

# Collect mitigation recommendations in text format
mitigation_text = []
mitigation_text.append("="*80)
mitigation_text.append("AGENTIC MITIGATION RECOMMENDATIONS (Attack-Type Specific)")
mitigation_text.append("="*80)
mitigation_text.append("")

for class_idx in range(num_classes):
    class_name = label_mapping_dict.get(class_idx, f"Class_{class_idx}")
    class_agent = agent_df[(agent_df['Class'] == class_name) & (agent_df['Is_Dominant'] == True)]
    if len(class_agent) > 0:
        row = class_agent.iloc[0]
        attack_type = class_name.upper()
        mitigation_text.append(f"\n{class_name} (Attack Type: {attack_type}):")
        mitigation_text.append(f"  Dominant Party: {row['Party']} ({row['Domain']})")
        mitigation_text.append(f"  Contribution: {row['Mean_contrib']*100:.2f}%")
        mitigation_text.append(f"\n{'='*80}")
        mitigation_text.append(f"{class_name} Attack - Mitigation Recommendations")
        mitigation_text.append(f"{'='*80}")
        mitigation_text.append(f"\nPRIMARY MITIGATION (Detected by dominant party):\n")
        mitigation_text.append(f"   {row['Party']} ({row['Mean_contrib']*100:.2f}% contribution) - PRIMARY MITIGATION:")
        mitigation_text.append(f"     Recommended Actions:")
        mitigation_text.append(format_action_readable(row['Suggested_Action']))
        
        # Also show actions from other parties for this attack type
        other_parties = agent_df[(agent_df['Class'] == class_name) & (agent_df['Is_Dominant'] == False)]
        if len(other_parties) > 0:
            mitigation_text.append(f"\nSUPPORTING MONITORING (Other parties):\n")
            for _, other_row in other_parties.iterrows():
                mitigation_text.append(f"   {other_row['Party']} ({other_row['Mean_contrib']*100:.2f}% contribution):")
                mitigation_text.append(f"     Action: {other_row['Suggested_Action']}")
                mitigation_text.append(f"     Status:  Monitoring mode - No active mitigation\n")

# Print to console
# # print("\n=== Agentic Mitigation Recommendations (Attack-Type Specific) ===")
# # for line in mitigation_text:
# print(line)

# Save to text file
mitigation_text_content = "\n".join(mitigation_text)
with open("outputs/vfl_shap_mitigation_summary.txt", "w", encoding="utf-8") as f:
    f.write(mitigation_text_content)

# # print(f"\nMitigation summary text file saved to: outputs/vfl_shap_mitigation_summary.txt")
# -----------------------------

In [ ]:
# 13. Predict on Sample Dataset and Save Results for LLM Analysis
# -----------------------------
# json and datetime are imported in Cell 0

# Helper function to partition and preprocess new samples (same as training)
def partition_and_preprocess_sample(sample_df, party1_features, party2_features, party3_features, 
                                     scaler1, scaler2, scaler3):
    """
    Partition a sample DataFrame into vertical parties and apply same preprocessing as training.
    
    Args:
        sample_df: DataFrame with raw features (must contain all features from party1/2/3_features)
        party1_features: List of feature names for party 1
        party2_features: List of feature names for party 2
        party3_features: List of feature names for party 3
        scaler1, scaler2, scaler3: Fitted StandardScaler objects from training
    
    Returns:
        Tuple of (X1, X2, X3) as torch tensors, same format as training data
    """
    # Extract features for each party (same vertical partitioning as training)
    X1_raw = sample_df[party1_features].values
    X2_raw = sample_df[party2_features].values
    X3_raw = sample_df[party3_features].values
    
    # Apply same preprocessing (scaling) as training
    X1_scaled = scaler1.transform(X1_raw)
    X2_scaled = scaler2.transform(X2_raw)
    X3_scaled = scaler3.transform(X3_raw)
    
    # Convert to torch tensors (same dtype as training)
    X1 = torch.tensor(X1_scaled, dtype=torch.float32)
    X2 = torch.tensor(X2_scaled, dtype=torch.float32)
    X3 = torch.tensor(X3_scaled, dtype=torch.float32)
    
    return X1, X2, X3

# Configuration
SAMPLE_SIZE = 50  # Number of samples to process (adjust as needed)
OUTPUT_DIR = "outputs"

print("="*80)
print("PREDICTING ON SAMPLE DATASET AND SAVING RESULTS")
print("="*80)
USE_NEW_DATASET= False

if USE_NEW_DATASET:
    # ============================================================
    # OPTION 1: Load your own dataset here
    # ============================================================
    # Example:
    # new_df = pd.read_csv("your_dataset.csv")
    # new_df = new_df.drop(columns=["Flow ID", "Src IP", "Dst IP", "Timestamp"], errors="ignore")
    # 
    # # Partition and preprocess using same feature lists and scalers as training
    # X1_new, X2_new, X3_new = partition_and_preprocess_sample(
    #     new_df, party1_features, party2_features, party3_features, scaler1, scaler2, scaler3
    # )
    # 
    # # Use new data for prediction
    # x_sample_parts = [X1_new, X2_new, X3_new]
    # y_sample = None  # If you have labels, set this
    # ============================================================
    raise NotImplementedError("Please set USE_NEW_DATASET=False or implement your dataset loading above")
else:
    # ============================================================
    # OPTION 2: Use preprocessed test set (current approach)
    # ============================================================
    print(f"\nUsing {SAMPLE_SIZE} samples from preprocessed test set...")
    x_sample_parts = x_test_parts
    y_sample = y_test

# Get sample indices
sample_indices = np.random.choice(len(x_sample_parts[0]), size=min(SAMPLE_SIZE, len(x_sample_parts[0])), replace=False)
sample_indices = sorted(sample_indices.tolist())  # Ensure it's a Python list, not numpy array

# Process each sample
results = []

print(f"\nProcessing {len(sample_indices)} samples...")
print("NOTE: Using same vertical partitioning and preprocessing as training")
for idx, sample_idx in enumerate(sample_indices):
    if (idx + 1) % 10 == 0:
        print(f"  Processed {idx + 1}/{len(sample_indices)} samples...")
    
    # Get sample data (already partitioned and preprocessed)
    X1_sample = x_sample_parts[0][sample_idx:sample_idx+1]
    X2_sample = x_sample_parts[1][sample_idx:sample_idx+1]
    X3_sample = x_sample_parts[2][sample_idx:sample_idx+1]
    
    # Get true label (if available)
    if y_sample is not None:
        true_label_idx = int(y_sample[sample_idx].item())  # Convert to Python int
        true_label = label_mapping_dict.get(true_label_idx, f"Class_{true_label_idx}")
    else:
        true_label_idx = None
        true_label = "UNKNOWN"
    
    # Predict with meta-model
    model.eval()
    meta_model.eval()
    with torch.no_grad():
        # Get embeddings
        embeddings = model.get_party_embeddings([X1_sample, X2_sample, X3_sample])
        X_meta_sample = torch.cat(embeddings, dim=1)
        
        # Predict
        logits = meta_model(X_meta_sample)
        probs = torch.softmax(logits, dim=1)
        predicted_class_idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0, predicted_class_idx].item()
    
    predicted_label = label_mapping_dict.get(predicted_class_idx, f"Class_{predicted_class_idx}")
    
    # Get all probabilities
    all_probs = {label_mapping_dict.get(i, f"Class_{i}"): float(probs[0, i].item()) 
                 for i in range(num_classes)}
    
    # Compute SHAP explanation (meta-level for party aggregation)
    X_meta_np = X_meta_sample.detach().cpu().numpy()
    shap_values_sample = explainer.shap_values(X_meta_np, nsamples=100)
    
    # Handle multi-class SHAP output
    if isinstance(shap_values_sample, list):
        shap_vals = shap_values_sample[predicted_class_idx][0]  # [192]
    else:
        shap_vals = shap_values_sample[0]  # [192]
    
    # Aggregate to party level
    embed_dim = 64
    party_shap_values = []
    for i in range(3):
        start_idx = i * embed_dim
        end_idx = (i + 1) * embed_dim
        party_contrib = float(np.sum(np.abs(shap_vals[start_idx:end_idx])))
        party_shap_values.append(party_contrib)
    
    # Calculate percentages
    total_shap = sum(party_shap_values)
    party_shap_pct = [p / total_shap if total_shap > 0 else 0 for p in party_shap_values]
    
    dominant_party_idx = int(np.argmax(party_shap_values))
    dominant_party = party_names[dominant_party_idx]
    
    # ============================================================
    # FEATURE-LEVEL SHAP CONTRIBUTIONS (NEW)
    # ============================================================
    # Create wrapper functions for each party to compute feature-level SHAP
    def create_party_predictor(party_idx, fixed_embeddings):
        """Create a prediction function for a specific party's features"""
        def party_predict(X_party_np):
            """Predict using party features, keeping other parties fixed"""
            X_party_t = torch.tensor(X_party_np, dtype=torch.float32)
            batch_size = X_party_t.shape[0]  # Get batch size from input
            
            model.eval()
            meta_model.eval()
            with torch.no_grad():
                # Get embedding for this party
                party_embedding = model.encoders[party_idx](X_party_t)  # [batch_size, embed_dim]
                
                # Combine with fixed embeddings from other parties
                # Fixed embeddings are from single sample, need to expand to match batch size
                all_embeddings = []
                for i, fixed_emb in enumerate(fixed_embeddings):
                    if i == party_idx:
                        # Use the new party embedding
                        all_embeddings.append(party_embedding)
                    else:
                        # Expand fixed embedding to match batch size
                        # fixed_emb is [1, embed_dim], expand to [batch_size, embed_dim]
                        # Use repeat() instead of expand() to ensure proper tensor (expand creates view)
                        if fixed_emb.shape[0] == 1:
                            expanded_emb = fixed_emb.repeat(batch_size, 1)  # [batch_size, embed_dim]
                        else:
                            # If already batch-sized, use as-is (shouldn't happen but handle it)
                            expanded_emb = fixed_emb[:batch_size] if fixed_emb.shape[0] >= batch_size else fixed_emb
                        all_embeddings.append(expanded_emb)
                
                X_meta_combined = torch.cat(all_embeddings, dim=1)  # [batch_size, embed_dim*3]
                
                # Predict
                logits = meta_model(X_meta_combined)
                probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            return probs
        return party_predict
    
    # Get fixed embeddings for other parties (for feature-level SHAP)
    model.eval()
    with torch.no_grad():
        all_embeddings_fixed = model.get_party_embeddings([X1_sample, X2_sample, X3_sample])
    
    # Compute feature-level SHAP for each party
    feature_shap_contributions = {}
    party_feature_lists = [party1_features, party2_features, party3_features]
    
    for party_idx in range(3):
        party_name = party_names[party_idx]
        party_feat_list = party_feature_lists[party_idx]
        
        # Get party's feature data (single sample)
        X_party_sample = [X1_sample, X2_sample, X3_sample][party_idx]
        X_party_np = X_party_sample.detach().cpu().numpy()  # [1, num_features]
        
        # Create background for this party (use a few samples from training data)
        # Use smaller background for faster computation
        bg_size_party = min(20, x_train_parts[party_idx].shape[0])
        bg_idx_party = torch.randperm(x_train_parts[party_idx].shape[0])[:bg_size_party]
        background_party = x_train_parts[party_idx][bg_idx_party].detach().cpu().numpy()  # [bg_size, num_features]
        
        # Create predictor for this party
        party_predict_fn = create_party_predictor(party_idx, all_embeddings_fixed)
        
        # Compute SHAP values for this party's features
        try:
            # Test the predictor function first
            test_output = party_predict_fn(X_party_np)
            if test_output.shape[0] != X_party_np.shape[0]:
                raise ValueError(f"Predictor output batch size mismatch: expected {X_party_np.shape[0]}, got {test_output.shape[0]}")
            
            explainer_party = shap.KernelExplainer(party_predict_fn, background_party)
            shap_values_party = explainer_party.shap_values(X_party_np, nsamples=100, silent=True)
            
            # Handle multi-class SHAP output
            if isinstance(shap_values_party, list):
                # Multi-class: shap_values_party is [num_classes, 1, num_features]
                shap_vals_party = shap_values_party[predicted_class_idx][0]  # [num_features]
            else:
                # Single output: shap_values_party is [1, num_features]
                shap_vals_party = shap_values_party[0] if len(shap_values_party.shape) > 1 else shap_values_party
            
            # Ensure we have the right shape
            shap_vals_party = np.array(shap_vals_party).flatten()
            
            # Calculate total absolute SHAP for this party (for percentage calculation)
            total_abs_shap_party = np.sum(np.abs(shap_vals_party))
            
            # Store feature-level contributions with percentages
            feature_contribs = {}
            for feat_idx, feat_name in enumerate(party_feat_list):
                if feat_idx < len(shap_vals_party):
                    shap_val = float(shap_vals_party[feat_idx])
                    abs_shap_val = float(np.abs(shap_val))
                    # Calculate percentage contribution within this party
                    pct_contrib = (abs_shap_val / total_abs_shap_party * 100.0) if total_abs_shap_party > 0 else 0.0
                    
                    feature_contribs[feat_name] = {
                        "shap_value": shap_val,
                        "abs_shap_value": abs_shap_val,
                        "pct_contribution": pct_contrib  # Percentage within this party
                    }
            
            feature_shap_contributions[party_name] = feature_contribs
            
        except Exception as e:
            # If SHAP computation fails, store empty dict and print detailed error
            import traceback
            error_msg = f"Feature-level SHAP failed for {party_name}: {str(e)}"
            print(f"  Warning: {error_msg}")
            if idx < 3:  # Only print traceback for first few samples to avoid spam
                print(f"  Traceback: {traceback.format_exc()}")
            feature_shap_contributions[party_name] = {}
    
    # Store result (ensure all values are JSON-serializable)
    result = {
        "sample_id": int(sample_idx),
        "true_label": true_label,
        "true_label_idx": int(true_label_idx) if true_label_idx is not None else None,
        "predicted_label": predicted_label,
        "predicted_label_idx": int(predicted_class_idx),
        "confidence": float(confidence),
        "all_probabilities": all_probs,
        "is_correct": (true_label == predicted_label) if true_label != "UNKNOWN" else None,
        "shap_explanation": {
            "party_contributions": {
                party_names[0]: float(party_shap_values[0]),
                party_names[1]: float(party_shap_values[1]),
                party_names[2]: float(party_shap_values[2])
            },
            "party_contributions_pct": {
                party_names[0]: float(party_shap_pct[0]),
                party_names[1]: float(party_shap_pct[1]),
                party_names[2]: float(party_shap_pct[2])
            },
            "dominant_party": dominant_party,
            "dominant_party_idx": int(dominant_party_idx),
            "dominant_contribution": float(party_shap_values[dominant_party_idx]),
            "dominant_contribution_pct": float(party_shap_pct[dominant_party_idx]),
            "total_contribution": float(total_shap),
            "feature_contributions": feature_shap_contributions  # NEW: Feature-level contributions
        },
        "timestamp": datetime.now().isoformat()
    }
    
    results.append(result)

# Convert to DataFrame for easier analysis
results_df = pd.DataFrame([
    {
        "sample_id": r["sample_id"],
        "true_label": r["true_label"],
        "predicted_label": r["predicted_label"],
        "confidence": r["confidence"],
        "is_correct": r["is_correct"],
        "dominant_party": r["shap_explanation"]["dominant_party"],
        "dominant_contribution_pct": r["shap_explanation"]["dominant_contribution_pct"]
    }
    for r in results
])

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"\nSaving results to {OUTPUT_DIR}/...")

# 1. Save detailed results as JSON (with timestamp)
detailed_json_filename = f"{OUTPUT_DIR}/sample_predictions_detailed_{timestamp}.json"
with open(detailed_json_filename, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# 2. Save summary as CSV
results_filename = f"{OUTPUT_DIR}/sample_predictions_summary_{timestamp}.csv"
results_df.to_csv(results_filename, index=False)

# 3. Save SHAP explanations separately (party-level)
shap_data = []
for r in results:
    shap_data.append({
        "sample_id": r["sample_id"],
        "predicted_label": r["predicted_label"],
        "party_1_contrib": r["shap_explanation"]["party_contributions"][party_names[0]],
        "party_1_contrib_pct": r["shap_explanation"]["party_contributions_pct"][party_names[0]],
        "party_2_contrib": r["shap_explanation"]["party_contributions"][party_names[1]],
        "party_2_contrib_pct": r["shap_explanation"]["party_contributions_pct"][party_names[1]],
        "party_3_contrib": r["shap_explanation"]["party_contributions"][party_names[2]],
        "party_3_contrib_pct": r["shap_explanation"]["party_contributions_pct"][party_names[2]],
        "dominant_party": r["shap_explanation"]["dominant_party"],
        "dominant_contribution": r["shap_explanation"]["dominant_contribution"],
        "dominant_contribution_pct": r["shap_explanation"]["dominant_contribution_pct"]
    })

shap_df = pd.DataFrame(shap_data)
shap_filename = f"{OUTPUT_DIR}/sample_shap_explanations_{timestamp}.csv"
shap_df.to_csv(shap_filename, index=False)

# 3b. Save feature-level SHAP contributions separately
feature_shap_data = []
feature_shap_filename = None
for r in results:
    sample_id = r["sample_id"]
    predicted_label = r.get("predicted_label", "UNKNOWN")
    feature_contribs = r["shap_explanation"].get("feature_contributions", {})
    
    for party_name, party_features in feature_contribs.items():
        for feat_name, feat_data in party_features.items():
            feature_shap_data.append({
                "sample_id": sample_id,
                "predicted_label": predicted_label,
                "party": party_name,
                "feature_name": feat_name,
                "shap_value": feat_data.get("shap_value", 0.0),
                "abs_shap_value": feat_data.get("abs_shap_value", 0.0),
                "pct_contribution": feat_data.get("pct_contribution", 0.0)  # Percentage within party
            })

if feature_shap_data:
    feature_shap_df = pd.DataFrame(feature_shap_data)
    feature_shap_filename = f"{OUTPUT_DIR}/sample_feature_shap_contributions_{timestamp}.csv"
    feature_shap_df.to_csv(feature_shap_filename, index=False)
    print(f"Feature-level SHAP contributions saved to {feature_shap_filename}")

# 4. Save metadata (ensure all values are JSON-serializable)
metadata = {
    "total_samples": int(len(results)),
    "sample_indices": [int(idx) for idx in sample_indices],  # Convert all to Python int
    "party_names": party_names,
    "label_mapping": {str(k): str(v) for k, v in label_mapping_dict.items()},
    "generation_timestamp": datetime.now().isoformat(),
    "model_info": {
        "num_classes": int(num_classes),
        "embed_dim": int(embed_dim) if 'embed_dim' in globals() else 64
    },
    "feature_partitioning": {
        "party1_features": party1_features,
        "party2_features": party2_features,
        "party3_features": party3_features,
        "party1_feature_count": len(party1_features),
        "party2_feature_count": len(party2_features),
        "party3_feature_count": len(party3_features)
    }
}

metadata_json_filename = f"{OUTPUT_DIR}/sample_predictions_metadata_{timestamp}.json"
with open(metadata_json_filename, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Print summary statistics
print("\n" + "="*80)
print("PREDICTION SUMMARY")
print("="*80)
print(f"Total samples processed: {len(results)}")
if y_sample is not None:
    accuracy = results_df['is_correct'].mean()
    print(f"Accuracy: {accuracy:.2%}")
else:
    print("Accuracy: N/A (no ground truth labels provided)")
print(f"\nPrediction distribution:")
print(results_df['predicted_label'].value_counts())
print(f"\nDominant party distribution:")
print(results_df['dominant_party'].value_counts())
print(f"\nFiles saved:")
print(f"  - {detailed_json_filename}")
print(f"  - {results_filename}")
print(f"  - {shap_filename}")
if feature_shap_data:
    print(f"  - {feature_shap_filename}")
print(f"  - {metadata_json_filename}")
print("\n" + "="*80)
print("Results saved! Ready for LLM-based action planning.")
print("\nNOTE: When using your own dataset, ensure:")
print("  1. Same feature names as training data")
print("  2. Use partition_and_preprocess_sample() function")
print("  3. Same vertical partitioning (party1/2/3_features)")
print("  4. Same preprocessing (scaler1/2/3 from training)")
print("="*80)

# -----------------------------